# Ìmọ̀lára · `smoke`

Pipeline check on Kaggle: real model, tiny data, 1 epoch. Not an experiment.

Runs `python -m src.train_transformer --config configs/smoke.yaml` for seeds 42 on a Kaggle T4 GPU.

* **Code:** [Hassan-Adelani-Luqman/imolara-yoruba-sentiment](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment) at commit [`ca42370`](https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment/tree/ca42370222d80c345648d7d78a08a1acb835aac5), cloned below; nothing is edited inside this notebook.
* **Data:** AfriSenti Yoruba (raw TSVs, pinned commit), downloaded by `src/data.py`.
* **Outputs:** `/kaggle/working/outputs/seed*/` (metrics, dev predictions, run info), downloaded to `results/kaggle/smoke/` in the repo.
* **Reproduce:** *Copy & Edit*, set Accelerator = GPU T4 and Internet = on, then *Run all*.

Generated by `scripts/kaggle_run.py --notebook`.

## 1 · Setup: clone the repo at the pinned commit, install extras, check the GPU

In [1]:
import glob, json, os, subprocess, sys

REPO, COMMIT = "https://github.com/Hassan-Adelani-Luqman/imolara-yoruba-sentiment.git", "ca42370222d80c345648d7d78a08a1acb835aac5"
CODE, OUT = "/tmp/imolara", "/kaggle/working/outputs"
CONFIG, SEEDS = "configs/smoke.yaml", [42]

subprocess.run(["git", "clone", "-q", REPO, CODE], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements-kaggle.txt"], check=True)
# Kaggle ships torchao 0.10, which peft >= 0.20 rejects at import time; we don't use it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/env.txt", "w") as f:            # exact package versions, for the report
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)

env = {**os.environ, "IMOLARA_GIT_COMMIT": COMMIT[:7], "PYTHONUNBUFFERED": "1"}
emb = glob.glob("/kaggle/input/**/w2v_yo_300.kv", recursive=True)
if emb:                                            # attached imolara-embeddings dataset (RNN runs)
    env["IMOLARA_EMB_DIR"] = os.path.dirname(emb[0])
    print("embeddings:", env["IMOLARA_EMB_DIR"])

GPU: Tesla T4 | torch 2.11.0+cu128


## 2 · Experiment configuration

In [2]:
print(open(f"{CODE}/{CONFIG}").read())

# Pipeline check on Kaggle: real model, tiny data, 1 epoch. Not an experiment.
exp_id: smoke
model_name: FacebookAI/xlm-roberta-base
train_subset: 200
epochs: 1
batch_size: 16
eval_diacritics: [original, no_tones, no_diacritics]
notes: pipeline smoke test



## 3 · Training: one cell per seed

The best epoch is chosen on *clean* dev macro-F1 (dev tweets that do not duplicate a training tweet).

In [3]:
def run(seed):
    cmd = [sys.executable, "-m", "src.train_transformer", "--config", CONFIG, "--seed", str(seed),
           "--output_dir", f"{OUT}/seed{seed}", *[]]
    proc = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        if "Warning" not in line and "warn(" not in line:
            print(line, end="")
    assert proc.wait() == 0, f"seed {seed} failed"

In [4]:
run(42)


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 2365.50it/s]
[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
  return supe

## 4 · Results across seeds

Macro-F1 is the primary metric; weighted-F1 is the official SemEval-2023 metric. `clean` excludes tweets that duplicate a training tweet; `no_tones` / `no_diacritics` score the same tweets with tone marks / all diacritics removed.

In [5]:
import pandas as pd

rows = [r for s in SEEDS for r in json.load(open(f"{OUT}/seed{s}/metrics.json"))]
df = pd.DataFrame(rows)
summary = (df.groupby(["split", "subset", "eval_diacritics"])[["macro_f1", "weighted_f1", "accuracy"]]
             .agg(["mean", "std"]).round(4))
summary

macro_f1     weighted_f1     accuracy    
                                 mean std        mean std     mean std
split subset eval_diacritics                                          
dev   all    no_diacritics     0.2804 NaN      0.2887 NaN   0.3589 NaN
             no_tones          0.2377 NaN      0.2347 NaN   0.2890 NaN
             original          0.1695 NaN      0.1471 NaN   0.2254 NaN
      clean  no_diacritics     0.2813 NaN      0.2893 NaN   0.3599 NaN
             no_tones          0.2356 NaN      0.2335 NaN   0.2867 NaN
             original          0.1726 NaN      0.1511 NaN   0.2289 NaN

In [6]:
per_seed = df[(df.subset == "clean") & (df.eval_diacritics == "original")][
    ["split", "seed", "macro_f1", "weighted_f1", "f1_negative", "f1_neutral", "f1_positive"]]
per_seed

,split,seed,macro_f1,weighted_f1,f1_negative,f1_neutral,f1_positive
1,dev,42,0.1726,0.1511,0.3247,0.0,0.1931
